### Importing Required Libraries

In [107]:
import numpy as np
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import os
from groq import Groq
import io
import time
from datetime import datetime
import gradio as gr
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Image as RLImage,
    Preformatted
)
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
import re

In [10]:
sales_data=pd.read_excel("Retail_sales_Data.xlsx")

### Data Cleaning and preprocessing

In [11]:
sales_data.head()

,transaction_id,customer_number,transaction_month,product_id,product_name,unit_price,volume_units,sales_amount,promo_offer
0,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01,P021,Dettol Antiseptic 250ml,305.70,10,3057.00,Yes
1,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01,P026,Kellogg's Corn Flakes 500g,405.20,5,2026.00,Yes
2,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01,P009,Lays Chips 200g,322.60,10,3226.00,Yes
3,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01,P037,Vim Dishwash Bar 200g,176.27,9,1586.43,Yes
4,a1e86962-46d7-4940-8759-b58c535b370b,75583263,2022-01,P026,Kellogg's Corn Flakes 500g,477.55,10,4775.50,No


In [12]:
sales_data.shape

(82950, 9)

In [13]:
sales_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82950 entries, 0 to 82949
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   transaction_id     82950 non-null  object 
 1   customer_number    82950 non-null  int64  
 2   transaction_month  82950 non-null  object 
 3   product_id         82950 non-null  object 
 4   product_name       82950 non-null  object 
 5   unit_price         82950 non-null  float64
 6   volume_units       82950 non-null  int64  
 7   sales_amount       82950 non-null  float64
 8   promo_offer        82950 non-null  object 
dtypes: float64(2), int64(2), object(5)
memory usage: 5.7+ MB


In [14]:
sales_data.isnull().sum()

transaction_id       0
customer_number      0
transaction_month    0
product_id           0
product_name         0
unit_price           0
volume_units         0
sales_amount         0
promo_offer          0
dtype: int64

In [15]:
sales_data.describe()

,customer_number,unit_price,volume_units,sales_amount
count,8.295000e+04,82950.000000,82950.000000,82950.000000
mean,4.973458e+07,273.435430,5.497601,1503.266978
std,2.866339e+07,162.663667,2.873561,1280.993174
min,3.277000e+03,29.220000,1.000000,29.570000
25%,2.432787e+07,127.200000,3.000000,504.490000
50%,4.978368e+07,243.490000,5.000000,1095.100000
75%,7.489914e+07,378.760000,8.000000,2219.805000
max,9.996990e+07,582.850000,10.000000,5828.500000


In [16]:
sales_data.duplicated().sum()

np.int64(0)

In [17]:
df=sales_data.copy()
df.head()

,transaction_id,customer_number,transaction_month,product_id,product_name,unit_price,volume_units,sales_amount,promo_offer
0,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01,P021,Dettol Antiseptic 250ml,305.70,10,3057.00,Yes
1,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01,P026,Kellogg's Corn Flakes 500g,405.20,5,2026.00,Yes
2,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01,P009,Lays Chips 200g,322.60,10,3226.00,Yes
3,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01,P037,Vim Dishwash Bar 200g,176.27,9,1586.43,Yes
4,a1e86962-46d7-4940-8759-b58c535b370b,75583263,2022-01,P026,Kellogg's Corn Flakes 500g,477.55,10,4775.50,No


In [18]:
df["transaction_month"] = pd.to_datetime(
    df["transaction_month"],
    format="%Y-%m",
    errors="coerce"
)

In [19]:
df.head()

,transaction_id,customer_number,transaction_month,product_id,product_name,unit_price,volume_units,sales_amount,promo_offer
0,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01-01,P021,Dettol Antiseptic 250ml,305.70,10,3057.00,Yes
1,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01-01,P026,Kellogg's Corn Flakes 500g,405.20,5,2026.00,Yes
2,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01-01,P009,Lays Chips 200g,322.60,10,3226.00,Yes
3,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01-01,P037,Vim Dishwash Bar 200g,176.27,9,1586.43,Yes
4,a1e86962-46d7-4940-8759-b58c535b370b,75583263,2022-01-01,P026,Kellogg's Corn Flakes 500g,477.55,10,4775.50,No


In [20]:
print(df["transaction_month"].isnull().sum())

0


In [21]:
df = df.drop_duplicates()

In [22]:
df.shape

(82950, 9)

In [23]:
numeric_cols = [
    "customer_number",
    "unit_price",
    "volume_units",
    "sales_amount"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [24]:
df = df.dropna(subset=numeric_cols)

In [25]:
df.shape

(82950, 9)

In [26]:
df = df[
    (df["unit_price"] > 0) &
    (df["volume_units"] > 0) &
    (df["sales_amount"] > 0)
]

In [27]:
text_cols = [
    "transaction_id",
    "product_id",
    "product_name",
    "promo_offer"
]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

In [28]:
df["year"] = df["transaction_month"].dt.year
df["month"] = df["transaction_month"].dt.month
df["month_name"] = df["transaction_month"].dt.strftime("%B")
df["quarter"] = df["transaction_month"].dt.quarter
df["year_month"] = df["transaction_month"].dt.strftime("%Y-%m")


In [29]:
df["calculated_sales"] = df["unit_price"] * df["volume_units"]

In [30]:
df["sales_difference"] = (
    df["sales_amount"] - df["calculated_sales"]
).round(2)

In [31]:
df["is_promo"] = np.where(
    df["promo_offer"].str.lower() == "no",
    0,
    1
)

In [32]:
df.head()

,transaction_id,customer_number,transaction_month,product_id,product_name,unit_price,volume_units,sales_amount,promo_offer,year,month,month_name,quarter,year_month,calculated_sales,sales_difference,is_promo
0,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01-01,P021,Dettol Antiseptic 250ml,305.70,10,3057.00,Yes,2022,1,January,1,2022-01,3057.00,0.0,1
1,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01-01,P026,Kellogg's Corn Flakes 500g,405.20,5,2026.00,Yes,2022,1,January,1,2022-01,2026.00,0.0,1
2,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01-01,P009,Lays Chips 200g,322.60,10,3226.00,Yes,2022,1,January,1,2022-01,3226.00,0.0,1
3,6ae6b99d-9158-43f2-ac7e-52f17620fc40,75583263,2022-01-01,P037,Vim Dishwash Bar 200g,176.27,9,1586.43,Yes,2022,1,January,1,2022-01,1586.43,0.0,1
4,a1e86962-46d7-4940-8759-b58c535b370b,75583263,2022-01-01,P026,Kellogg's Corn Flakes 500g,477.55,10,4775.50,No,2022,1,January,1,2022-01,4775.50,0.0,0


### Exploratory Data Analysis (EDA)

In [33]:
print("Unique Products:", df["product_name"].nunique())
print("Unique Customers:", df["customer_number"].nunique())
print("Unique Promotions:", df["promo_offer"].nunique())

Unique Products: 41
Unique Customers: 2000
Unique Promotions: 2


In [34]:
total_sales = df["sales_amount"].sum()
total_sales

np.float64(124695995.83000001)

In [35]:
total_units = df["volume_units"].sum()
total_units

np.int64(456026)

In [36]:
avg_order_value = df["sales_amount"].mean()
avg_order_value

np.float64(1503.2669780590718)

In [37]:
monthly_sales = (
    df.groupby("year_month")["sales_amount"]
    .sum()
    .reset_index()
)

monthly_sales

,year_month,sales_amount
0,2022-01,1641784.46
1,2022-02,3006115.15
2,2022-03,4214472.09
3,2022-04,5287652.14
4,2022-05,5536292.74
5,2022-06,5840271.84
6,2022-07,5842066.12
7,2022-08,6165023.84
8,2022-09,6191852.10
9,2022-10,6028041.60


In [38]:
top_products = (
    df.groupby("product_name")["sales_amount"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products

product_name
Godrej Shampoo 200ml          6291047.29
Milky Mist Paneer 200g        6278408.60
Santoor Soap 75g              6029465.11
Pepsodent Combo Pack          5902345.04
Dove Soap 75g                 5534074.21
Red Label Tea 250g            5374852.36
Palmolive Shampoo 180ml       5312118.65
Kellogg's Corn Flakes 500g    5098624.12
Amul Butter 100g              4783230.37
Dabur Honey 500g              4732191.67
Name: sales_amount, dtype: float64

In [39]:
top_products_units = (
    df.groupby("product_name")["volume_units"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products_units

product_name
Cinthol Soap 100g                11584
Patanjali Aloe Vera Gel 100ml    11563
Amul Butter 100g                 11459
Tata Salt 1kg                    11446
Surf Excel Detergent 1kg         11438
Dove Soap 75g                    11412
Pepsi Cola 2L Bottle             11386
Amul Cheese Slice 400g           11384
Cadbury Dairy Milk 50g           11380
Fogg Perfume 50ml                11373
Name: volume_units, dtype: int64

In [40]:
promo_analysis = (
    df.groupby("promo_offer")["sales_amount"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
)

promo_analysis

,sum,mean,count
promo_offer,,,
No,1.037290e+08,1568.277855,66142
Yes,2.096696e+07,1247.439430,16808


In [41]:
sales_by_year = (
    df.groupby("year")["sales_amount"]
    .sum()
)

sales_by_year

year
2022    61870168.90
2023    62825826.93
Name: sales_amount, dtype: float64

In [42]:
sales_by_quarter = (
    df.groupby(["year", "quarter"])["sales_amount"]
    .sum()
)

sales_by_quarter

year  quarter
2022  1           8862371.70
      2          16664216.72
      3          18198942.06
      4          18144638.42
2023  1          18741051.29
      2          18382632.09
      3          16893840.88
      4           8808302.67
Name: sales_amount, dtype: float64

In [43]:
monthly_sales["growth_pct"] = (
    monthly_sales["sales_amount"]
    .pct_change() * 100
)

monthly_sales

,year_month,sales_amount,growth_pct
0,2022-01,1641784.46,NaN
1,2022-02,3006115.15,83.100475
2,2022-03,4214472.09,40.196629
3,2022-04,5287652.14,25.464163
4,2022-05,5536292.74,4.702287
5,2022-06,5840271.84,5.490662
6,2022-07,5842066.12,0.030723
7,2022-08,6165023.84,5.528142
8,2022-09,6191852.10,0.435169
9,2022-10,6028041.60,-2.645582


In [44]:
top_customers = (
    df.groupby("customer_number")["sales_amount"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_customers

customer_number
12686676    160586.26
91152785    148504.96
18268477    148283.43
36222929    146788.94
38305911    145738.45
12663393    138943.84
84545613    135828.45
86755181    135419.58
45321518    134962.20
62654175    134698.99
Name: sales_amount, dtype: float64

In [45]:
df["unit_price"].describe()

count    82950.000000
mean       273.435430
std        162.663667
min         29.220000
25%        127.200000
50%        243.490000
75%        378.760000
max        582.850000
Name: unit_price, dtype: float64

In [46]:
correlation = df[
    ["unit_price", "volume_units", "sales_amount"]
].corr()

correlation

,unit_price,volume_units,sales_amount
unit_price,1.000000,0.000060,0.698937
volume_units,0.000060,1.000000,0.614519
sales_amount,0.698937,0.614519,1.000000


### Loading Cleaned Data into SQLite Database

In [47]:
DB_NAME = "sales.db"
TABLE_NAME = "sales_data"

In [48]:
# Creating SQLite connection
conn = sqlite3.connect(DB_NAME)

df.to_sql(
    TABLE_NAME,
    conn,
    if_exists="replace",   
    index=False
)

conn.close()

print("Data loaded into SQLite successfully!")

Data loaded into SQLite successfully!


In [49]:
def run_query(sql):
    conn = sqlite3.connect(DB_NAME)
    result = pd.read_sql_query(sql, conn)
    conn.close()
    return result

In [50]:
def get_total_sales():
    sql = """
        SELECT ROUND(SUM(sales_amount), 2) AS total_sales
        FROM sales_data
    """
    result = run_query(sql)
    total = result.loc[0, "total_sales"]
    return f"Total sales are ₹{total:,.2f}"

In [51]:
def get_total_units():
    sql = """
        SELECT SUM(volume_units) AS total_units
        FROM sales_data
    """
    result = run_query(sql)
    units = int(result.loc[0, "total_units"])
    return f"Total units sold are {units:,}"

In [52]:
def get_average_order_value():
    sql = """
        SELECT ROUND(AVG(sales_amount), 2) AS avg_order_value
        FROM sales_data
    """
    result = run_query(sql)
    avg_value = result.loc[0, "avg_order_value"]
    return f"Average order value is ₹{avg_value:,.2f}"

In [53]:
def get_top_products():
    sql = """
        SELECT product_name,
               ROUND(SUM(sales_amount), 2) AS revenue
        FROM sales_data
        GROUP BY product_name
        ORDER BY revenue DESC
        LIMIT 10
    """
    result = run_query(sql)
    return result

In [54]:
def get_monthly_sales():
    sql = """
        SELECT year_month,
               ROUND(SUM(sales_amount), 2) AS total_sales
        FROM sales_data
        GROUP BY year_month
        ORDER BY year_month
    """
    result = run_query(sql)
    return result

In [55]:
def get_promotion_impact():
    sql = """
        SELECT promo_offer,
               ROUND(SUM(sales_amount), 2) AS total_sales,
               ROUND(AVG(sales_amount), 2) AS avg_sales,
               COUNT(*) AS transactions
        FROM sales_data
        GROUP BY promo_offer
        ORDER BY total_sales DESC
    """
    result = run_query(sql)
    return result

In [56]:
def get_top_customers():
    sql = """
        SELECT customer_number,
               ROUND(SUM(sales_amount), 2) AS total_sales
        FROM sales_data
        GROUP BY customer_number
        ORDER BY total_sales DESC
        LIMIT 10
    """
    result = run_query(sql)
    return result

In [57]:
def get_product_summary(product_name):
    sql = f"""
        SELECT product_name,
               SUM(volume_units) AS units_sold,
               ROUND(SUM(sales_amount), 2) AS revenue,
               ROUND(AVG(unit_price), 2) AS avg_price
        FROM sales_data
        WHERE LOWER(product_name) = LOWER('{product_name}')
        GROUP BY product_name
    """
    result = run_query(sql)
    return result

In [58]:
def compare_products(product1, product2):
    sql = f"""
        SELECT product_name,
               ROUND(SUM(sales_amount), 2) AS revenue,
               SUM(volume_units) AS units_sold
        FROM sales_data
        WHERE LOWER(product_name) IN (
            LOWER('{product1}'),
            LOWER('{product2}')
        )
        GROUP BY product_name
        ORDER BY revenue DESC
    """
    result = run_query(sql)
    return result

In [59]:
def get_best_month():
    sql = """
        SELECT year_month,
               ROUND(SUM(sales_amount), 2) AS total_sales
        FROM sales_data
        GROUP BY year_month
        ORDER BY total_sales DESC
        LIMIT 1
    """
    result = run_query(sql)
    row = result.iloc[0]
    return f"Best month was {row['year_month']} with sales of ₹{row['total_sales']:,.2f}"

In [60]:
print(get_total_sales())
print(get_total_units())
print(get_average_order_value())
print(get_best_month())

print(get_top_products())
print(get_promotion_impact())

Total sales are ₹124,695,995.83
Total units sold are 456,026
Average order value is ₹1,503.27
Best month was 2023-03 with sales of ₹6,481,715.86
                 product_name     revenue
0        Godrej Shampoo 200ml  6291047.29
1      Milky Mist Paneer 200g  6278408.60
2            Santoor Soap 75g  6029465.11
3        Pepsodent Combo Pack  5902345.04
4               Dove Soap 75g  5534074.21
5          Red Label Tea 250g  5374852.36
6     Palmolive Shampoo 180ml  5312118.65
7  Kellogg's Corn Flakes 500g  5098624.12
8            Amul Butter 100g  4783230.37
9            Dabur Honey 500g  4732191.67
  promo_offer   total_sales  avg_sales  transactions
0          No  1.037290e+08    1568.28         66142
1         Yes  2.096696e+07    1247.44         16808


### Building the Analytics Engine

In [61]:
def detect_intent(question):
    q = question.lower().strip()

    # Total Sales
    if "total sales" in q:
        return {
            "intent": "total_sales",
            "params": {}
        }

    # Total Units
    elif "total units" in q or "units sold" in q:
        return {
            "intent": "total_units",
            "params": {}
        }

    # Average Order Value
    elif "average order value" in q or "avg order value" in q:
        return {
            "intent": "average_order_value",
            "params": {}
        }

    # Monthly Trend
    elif (
        "monthly trend" in q or
        "sales trend" in q or
        "monthly sales" in q
    ):
        return {
            "intent": "monthly_sales_trend",
            "params": {}
        }

    # Top Products
    elif (
        "top products" in q or
        "best products" in q
    ):
        return {
            "intent": "top_products",
            "params": {}
        }

    # Promotion Impact
    elif (
        "promotion" in q or
        "promo impact" in q
    ):
        return {
            "intent": "promotion_impact",
            "params": {}
        }

    # Top Customers
    elif (
        "top customers" in q or
        "best customers" in q
    ):
        return {
            "intent": "top_customers",
            "params": {}
        }

    # Best Month
    elif (
        "best month" in q or
        "highest sales month" in q
    ):
        return {
            "intent": "best_month",
            "params": {}
        }

    # Compare Products
    elif "compare" in q and "and" in q:
        import re

        match = re.search(
            r"compare\\s+(.*?)\\s+and\\s+(.*)",
            q
        )

        if match:
            product1 = match.group(1).strip().title()
            product2 = match.group(2).strip().title()

            return {
                "intent": "compare_products",
                "params": {
                    "product1": product1,
                    "product2": product2
                }
            }

    # Product Summary
    elif "product summary for" in q:
        product = q.replace(
            "product summary for",
            ""
        ).strip().title()

        return {
            "intent": "product_summary",
            "params": {
                "product_name": product
            }
        }

    # Unknown
    return {
        "intent": "unknown",
        "params": {}
    }

In [62]:
def format_dataframe(df, title="Results"):
    # If DataFrame is empty
    if df.empty:
        return "No data found."

    lines = [f"{title}:\n"]

    # Iterate through each row
    for index, row in df.iterrows():
        row_parts = []

        for col in df.columns:
            value = row[col]

            # Format numeric values nicely
            if isinstance(value, (int, float)):
                if "sales" in col.lower() or "revenue" in col.lower() or "price" in col.lower():
                    formatted_value = f"₹{value:,.2f}"
                else:
                    # Remove decimal if whole number
                    if float(value).is_integer():
                        formatted_value = f"{int(value):,}"
                    else:
                        formatted_value = f"{value:,.2f}"
            else:
                formatted_value = str(value)

            row_parts.append(f"{col}: {formatted_value}")

        lines.append(f"{index + 1}. " + " | ".join(row_parts))

    return "\n".join(lines)

##### Generating Charts for Visualization Questions

In [63]:
CHART_DIR = "charts"
os.makedirs(CHART_DIR, exist_ok=True)

In [64]:
def create_monthly_sales_chart():
    # Get aggregated data
    df_monthly = get_monthly_sales()

    if df_monthly.empty:
        return None

    # Convert year_month to datetime for correct sorting
    df_monthly["year_month"] = pd.to_datetime(
        df_monthly["year_month"]
    )

    # Create chart
    plt.figure(figsize=(10, 5))
    plt.plot(
        df_monthly["year_month"],
        df_monthly["total_sales"],
        marker="o"
    )

    plt.title("Monthly Sales Trend")
    plt.xlabel("Month")
    plt.ylabel("Sales Amount")
    plt.xticks(rotation=45)
    plt.tight_layout()

    # Save chart
    chart_path = os.path.join(
        CHART_DIR,
        "monthly_sales.png"
    )

    plt.savefig(chart_path)
    plt.close()

    return chart_path

In [65]:
def create_top_products_chart():
    df_top = get_top_products()

    if df_top.empty:
        return None

    plt.figure(figsize=(10, 6))
    plt.barh(
        df_top["product_name"],
        df_top["revenue"]
    )

    plt.title("Top 10 Products by Revenue")
    plt.xlabel("Revenue")
    plt.tight_layout()

    chart_path = os.path.join(
        CHART_DIR,
        "top_products.png"
    )

    plt.savefig(chart_path)
    plt.close()

    return chart_path

##### Adding Generative AI Explanations

In [66]:
client = Groq(api_key="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")

In [67]:
GROQ_MODEL = "openai/gpt-oss-120b"

In [68]:
def generate_llm_response(question, analytics_answer):
    """
    Sends the analytics result to Groq and returns a business-friendly explanation.
    If the API call fails, returns the original analytics answer.
    """

    prompt = f"""
You are a retail sales analyst.

User Question:
{question}

Analytics Result:
{analytics_answer}

Please provide:
1. A concise explanation in business language.
2. Key insights.
3. Suggested follow-up questions.

Keep the response easy to understand and professional.
"""

    try:
        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2,
            max_tokens=500
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        print("Groq API Error:", e)
        return analytics_answer

In [69]:
def handle_question(question):
    # Step 1: Detect user intent
    result = detect_intent(question)
    intent = result["intent"]
    params = result["params"]

    chart_path = None

    # Step 2: Execute analytics function
    if intent == "total_sales":
        analytics_result = get_total_sales()

    elif intent == "total_units":
        analytics_result = get_total_units()

    elif intent == "average_order_value":
        analytics_result = get_average_order_value()

    elif intent == "monthly_sales_trend":
        df_result = get_monthly_sales()
        analytics_result = format_dataframe(
            df_result,
            "Monthly Sales Trend"
        )
        chart_path = create_monthly_sales_chart()

    elif intent == "top_products":
        df_result = get_top_products()
        analytics_result = format_dataframe(
            df_result,
            "Top Products by Revenue"
        )
        chart_path = create_top_products_chart()

    elif intent == "promotion_impact":
        df_result = get_promotion_impact()
        analytics_result = format_dataframe(
            df_result,
            "Promotion Impact Analysis"
        )

    elif intent == "top_customers":
        df_result = get_top_customers()
        analytics_result = format_dataframe(
            df_result,
            "Top Customers"
        )

    elif intent == "best_month":
        analytics_result = get_best_month()

    elif intent == "compare_products":
        df_result = compare_products(
            params["product1"],
            params["product2"]
        )
        analytics_result = format_dataframe(
            df_result,
            "Product Comparison"
        )

    elif intent == "product_summary":
        df_result = get_product_summary(
            params["product_name"]
        )
        analytics_result = format_dataframe(
            df_result,
            "Product Summary"
        )

    else:
        analytics_result = (
            "I can help with questions like:\n"
            "- What are total sales?\n"
            "- Show monthly sales trend\n"
            "- Top products\n"
            "- Promotion impact\n"
            "- Compare Shampoo and Soap"
        )

    # Step 3: Generate business-friendly explanation using Groq
    final_answer = generate_llm_response(
        question,
        analytics_result
    )

    # Step 4: Return both answer and chart path
    return {
        "answer": final_answer,
        "chart_path": chart_path
    }

### Creating the Chat UI Widgets

In [112]:
# CHAT FUNCTION
def chat_fn(message, history):
    global chat_history

    # Run your existing chatbot logic
    result = handle_question(message)

    answer = result["answer"]
    chart_path = result["chart_path"]

    # Save to chat history for PDF report
    chat_history.append({
        "question": message,
        "answer": answer,
        "chart_path": chart_path
    })

    return answer, chart_path


# WRAPPER FUNCTION TO SHOW/HIDE CHART
def chat_wrapper(message, history):
    answer, chart_path = chat_fn(message, history)

    # If chart exists, display it
    if chart_path and os.path.exists(chart_path):
        chart_update = gr.update(
            value=chart_path,
            visible=True
        )
    else:
        # Hide chart if no chart exists
        chart_update = gr.update(
            value=None,
            visible=False
        )

    return answer, chart_update

# REFRESH DATA FUNCTION
def refresh_data_ui():
    """
    Refreshes the source data and returns a success or error message
    that will be displayed in the Refresh Status textbox.
    """
    try:
        # refresh_data() should return something like:
        # "Data refreshed successfully! 82,950 rows loaded."
        message = refresh_data()

        # If refresh_data() returns nothing, provide a default message
        if message is None or str(message).strip() == "":
            message = "Data refreshed successfully!"

        return message

    except Exception as e:
        return f"Error while refreshing data: {str(e)}"

# PDF REPORT FUNCTION
def clean_text_for_pdf(text):
    """
    Convert markdown/HTML-rich text into plain text that
    ReportLab can safely render.
    """
    if text is None:
        return ""

    text = str(text)

    # Remove HTML tags such as <br>
    text = re.sub(r"<[^>]+>", "\n", text)

    # Remove markdown bold/italic markers
    text = text.replace("**", "")
    text = text.replace("__", "")
    text = text.replace("*", "")

    # Convert markdown table separators to plain text
    text = text.replace("|", " ")
    text = text.replace("----", "")

    # Normalize line breaks
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def generate_pdf_report():
    """
    Generate PDF report containing:
    - Questions
    - Answers
    - Charts
    """
    global chat_history

    if not chat_history:
        return None

    # File name
    filename = f"retail_chat_report_{int(time.time())}.pdf"
    filepath = os.path.abspath(filename)

    # PDF document
    doc = SimpleDocTemplate(
        filepath,
        pagesize=A4,
        rightMargin=40,
        leftMargin=40,
        topMargin=40,
        bottomMargin=40
    )

    styles = getSampleStyleSheet()
    story = []

    # Title
    story.append(
        Paragraph(
            "<b>Retail Sales Chatbot Report</b>",
            styles["Title"]
        )
    )
    story.append(Spacer(1, 20))

    # Chat History
    for i, item in enumerate(chat_history, start=1):

        question = clean_text_for_pdf(item.get("question", ""))
        answer = clean_text_for_pdf(item.get("answer", ""))
        chart_path = item.get("chart_path")

        # Question heading
        story.append(
            Paragraph(
                f"<b>Question {i}:</b> {question}",
                styles["BodyText"]
            )
        )
        story.append(Spacer(1, 6))

        # Answer as preformatted text (safer than Paragraph)
        story.append(
            Preformatted(
                f"Answer:\n{answer}",
                styles["Code"]
            )
        )
        story.append(Spacer(1, 12))

        # Add chart if available
        if (
            chart_path
            and os.path.exists(chart_path)
        ):
            try:
                img = RLImage(
                    chart_path,
                    width=500,
                    height=250
                )
                story.append(img)
                story.append(Spacer(1, 12))
            except Exception:
                pass

        story.append(Spacer(1, 20))

    # Build PDF
    doc.build(story)

    return filepath

# BUILD UI
with gr.Blocks() as demo:
    gr.Markdown("# GenAI Retail Sales Chatbot")
    gr.Markdown(
        "Ask natural language questions about your retail sales data."
    )

    # Chart Output (hidden initially)
    chart_output = gr.Image(
        label="Chart",
        type="filepath",
        visible=False
    )

    # Chat Interface
    chat = gr.ChatInterface(
        fn=chat_wrapper,
        additional_outputs=[chart_output],
        examples=[
            "What are total sales?",
            "What are total units sold?",
            "What is the average order value?",
            "Show monthly sales trend",
            "Top products",
            "Promotion impact",
            "Top customers",
            "Best month",
            "Compare Shampoo and Soap"
        ],
        title=None
    )

    # Refresh Data Section
    gr.Markdown("## Refresh Sales Data")

    refresh_btn = gr.Button("Refresh Data")
    refresh_status = gr.Textbox(
        label="Refresh Status",
        interactive=False
    )

    refresh_btn.click(
        fn=refresh_data_ui,
        inputs=[],
        outputs=refresh_status
    )

    # Download Report Section
    gr.Markdown("## Download Chat Report")

    download_btn = gr.Button("Generate PDF Report")
    report_file = gr.File(label="Download Report")

    download_btn.click(
        fn=generate_pdf_report,
        inputs=[],
        outputs=report_file
    )

# LAUNCH APPLICATION
demo.launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


### Testing and Validation

In [78]:
print(df[["transaction_month", "year_month"]].head())
print(df["transaction_month"].min())
print(df["transaction_month"].max())

  transaction_month year_month
0        2022-01-01    2022-01
1        2022-01-01    2022-01
2        2022-01-01    2022-01
3        2022-01-01    2022-01
4        2022-01-01    2022-01
2022-01-01 00:00:00
2023-12-01 00:00:00


In [79]:
run_query("SELECT COUNT(*) AS row_count FROM sales_data")

,row_count
0,82950


In [80]:
print(get_total_sales())
print(get_total_units())
print(get_average_order_value())
print(get_best_month())

Total sales are ₹124,695,995.83
Total units sold are 456,026
Average order value is ₹1,503.27
Best month was 2023-03 with sales of ₹6,481,715.86


In [81]:
print(get_top_products().head())
print(get_monthly_sales().head())
print(get_promotion_impact().head())
print(get_top_customers().head())

             product_name     revenue
0    Godrej Shampoo 200ml  6291047.29
1  Milky Mist Paneer 200g  6278408.60
2        Santoor Soap 75g  6029465.11
3    Pepsodent Combo Pack  5902345.04
4           Dove Soap 75g  5534074.21
  year_month  total_sales
0    2022-01   1641784.46
1    2022-02   3006115.15
2    2022-03   4214472.09
3    2022-04   5287652.14
4    2022-05   5536292.74
  promo_offer   total_sales  avg_sales  transactions
0          No  1.037290e+08    1568.28         66142
1         Yes  2.096696e+07    1247.44         16808
   customer_number  total_sales
0         12686676    160586.26
1         91152785    148504.96
2         18268477    148283.43
3         36222929    146788.94
4         38305911    145738.45


In [82]:
test_questions = [
    "What are total sales?",
    "Top products",
    "Show monthly sales trend",
    "Promotion impact",
    "Top customers",
    "Best month",
    "Compare Shampoo and Soap",
    "Product summary for Shampoo"
]

for q in test_questions:
    print("Question:", q)
    print(detect_intent(q))
    print("-" * 50)

Question: What are total sales?
{'intent': 'total_sales', 'params': {}}
--------------------------------------------------
Question: Top products
{'intent': 'top_products', 'params': {}}
--------------------------------------------------
Question: Show monthly sales trend
{'intent': 'monthly_sales_trend', 'params': {}}
--------------------------------------------------
Question: Promotion impact
{'intent': 'promotion_impact', 'params': {}}
--------------------------------------------------
Question: Top customers
{'intent': 'top_customers', 'params': {}}
--------------------------------------------------
Question: Best month
{'intent': 'best_month', 'params': {}}
--------------------------------------------------
Question: Compare Shampoo and Soap
{'intent': 'unknown', 'params': {}}
--------------------------------------------------
Question: Product summary for Shampoo
{'intent': 'product_summary', 'params': {'product_name': 'Shampoo'}}
------------------------------------------------

In [83]:
print(format_dataframe(get_top_products().head(), "Top Products"))

Top Products:

1. product_name: Godrej Shampoo 200ml | revenue: ₹6,291,047.29
2. product_name: Milky Mist Paneer 200g | revenue: ₹6,278,408.60
3. product_name: Santoor Soap 75g | revenue: ₹6,029,465.11
4. product_name: Pepsodent Combo Pack | revenue: ₹5,902,345.04
5. product_name: Dove Soap 75g | revenue: ₹5,534,074.21


In [84]:
print(create_monthly_sales_chart())
print(create_top_products_chart())

charts\monthly_sales.png
charts\top_products.png


In [85]:
import os
print(os.path.exists(create_monthly_sales_chart()))

True


In [86]:
print(generate_llm_response(
    "What are total sales?",
    "Total sales are ₹124,701,499.82"
))

**1. Concise Explanation**  
The total sales figure for the period under review is **₹124,701,499.82**. This amount represents the aggregate revenue generated from all transactions across the retail portfolio during the reporting window.

**2. Key Insights**  
- **Scale of Operations:** Crossing the ₹124 million mark indicates a sizable market presence and a healthy volume of transactions.  
- **Revenue Benchmark:** This figure can serve as a baseline for comparing performance month‑over‑month, quarter‑over‑quarter, or against prior fiscal years.  
- **Profitability Leverage:** With total sales known, you can now calculate gross margin, operating margin, and net profit to assess how efficiently the business converts sales into earnings.  
- **Resource Allocation:** High sales volume may justify continued or increased investment in inventory, staffing, and marketing to sustain growth.  
- **Risk Indicator:** If this total is significantly lower than historical averages or targets, it ma

In [87]:
questions = [
    "What are total sales?",
    "Show monthly sales trend",
    "Top products",
    "Promotion impact"
]

for q in questions:
    result = handle_question(q)
    print("Question:", q)
    print("Answer:", result["answer"][:300])
    print("Chart:", result["chart_path"])
    print("=" * 80)

Question: What are total sales?
Answer: **1. Concise Explanation (Business Language)**  
The company generated total sales of **₹124,695,995.83** for the period under review. This figure represents the aggregate revenue from all transactions before any deductions such as discounts, returns, or taxes.

**2. Key Insights**  
| Insight | Why
Chart: None
Question: Show monthly sales trend
Answer: **1. Concise Business Explanation**

The monthly sales data shows a clear growth trajectory in 2022, peaking in August 2022 at ₹6.17 M, followed by a relatively stable plateau through the end of the year. In 2023 the business started the year at a similar level to late‑2022, but sales began a steady
Chart: charts\monthly_sales.png
Question: Top products
Answer: **1. Concise Explanation (Business Language)**  
The list shows the ten highest‑revenue products in the current reporting period. Together they generate the bulk of sales value, with the top three items (Godrej Shampoo 200 ml, Milky Mist Pa

In [88]:
print(refresh_data())

Refreshing data...
Data refresh completed!
Rows loaded: 82950
None


In [89]:
print(generate_pdf_report())

C:\Users\nihar\Documents\ML Projects\Capstone 2\retail_chat_report_1779089569.pdf


### Feedback Datastructure Example

In [93]:
#After completing development and testing, the final step is to gather structured feedback from users and convert it into actionable UI/UX improvements.

In [90]:
feedback = [
    {
        "user": "Analyst 1",
        "ease_of_use": 5,
        "response_quality": 4,
        "chart_usefulness": 5,
        "overall": 5,
        "comments": "Very intuitive and helpful."
    },
    {
        "user": "Manager 1",
        "ease_of_use": 4,
        "response_quality": 5,
        "chart_usefulness": 4,
        "overall": 4,
        "comments": "Would like more KPIs on the homepage."
    }
]

In [91]:
feedback_df = pd.DataFrame(feedback)

print(feedback_df.mean(numeric_only=True))

ease_of_use         4.5
response_quality    4.5
chart_usefulness    4.5
overall             4.5
dtype: float64
